In [1]:
import sys
sys.path.append("../")

from src.ingestion import load_config, load_file
from src.profiler import profile_dataframe
import json

config = load_config("../config/pipeline_config.yaml")
print("✅ Config loaded:", config["pipeline"]["name"])

df, metadata = load_file("../data/raw/tshirts.csv")
print("\n📊 Metadata:")
print(json.dumps(metadata, indent=2))

profile = profile_dataframe(df, config)
print(f"\n🏥 Overall Quality Score: {profile['overall_quality_score']}%")

print("\n🔍 Column-wise Health:")
for col, details in profile["column_profiles"].items():
    print(f"  {col}: {details['severity']} | Null%: {details['null_percentage']*100}% | Type: {details['data_type']}")

✅ Config loaded: DataSentinel

📊 Metadata:
{
  "file_name": "tshirts.csv",
  "file_format": "csv",
  "file_size_kb": 220.91,
  "load_time_seconds": 0.027088,
  "row_count": 39,
  "column_count": 24,
  "columns": [
    "product_id",
    "title",
    "product_description",
    "rating",
    "ratings_count",
    "initial_price",
    "discount",
    "final_price",
    "currency",
    "images",
    "delivery_options",
    "product_details",
    "breadcrumbs",
    "product_specifications",
    "amount_of_stars",
    "what_customers_said",
    "seller_name",
    "sizes",
    "videos",
    "seller_information",
    "variations",
    "best_offer",
    "more_offers",
    "category"
  ],
  "loaded_at": "2026-07-03 16:59:32.399054"
}

🏥 Overall Quality Score: 87.5%

🔍 Column-wise Health:
  product_id: OK | Null%: 0.0% | Type: int64
  title: OK | Null%: 0.0% | Type: object
  product_description: OK | Null%: 0.0% | Type: object
  rating: OK | Null%: 0.0% | Type: float64
  ratings_count: OK | Null%: 

In [2]:
from src.validator import apply_validation_rules
from src.quarantine import quarantine_bad_rows, save_quarantine

# Validation rules apply karo
violations, summary = apply_validation_rules(df, config)

print("📋 Validation Summary:")
print(json.dumps(summary, indent=2))

print("\n⚠️ Sample Violations:")
for v in violations[:5]:  # Pehli 5 violations dikhao
    print(f"  Row {v['row_index']} | {v['rule_name']} | {v['column']} = {v['actual_value']} | {v['severity']}")

# Quarantine karo
clean_df, quarantined_df = quarantine_bad_rows(df, violations, config)

print(f"\n✅ Clean rows: {len(clean_df)}")
print(f"🚨 Quarantined rows: {len(quarantined_df)}")

if not quarantined_df.empty:
    print("\n🔍 Quarantine Preview:")
    print(quarantined_df[["product_id", "quarantine_reason"]].head())

    # Save karo file mein
    saved_path = save_quarantine(quarantined_df, config, metadata["file_name"])
    print(f"\n💾 Quarantine saved at: {saved_path}")

📋 Validation Summary:
{
  "total_rows_checked": 39,
  "total_violations": 0,
  "critical_violations": 0,
  "warning_violations": 0
}

⚠️ Sample Violations:

✅ Clean rows: 39
🚨 Quarantined rows: 0


In [3]:
print("Violations found:", len(violations_test))
print(violations_test)
print()
print("Rating dtype:", test_df["rating"].dtype)
print("Initial_price dtype:", test_df["initial_price"].dtype)
print()
print(test_df.iloc[-1][["rating", "initial_price"]])

NameError: name 'violations_test' is not defined

In [5]:
print(config.get("validation_rules", "NO RULES FOUND"))

[{'name': 'no_negative_price', 'column': 'initial_price', 'condition': 'less_than', 'value': 0, 'severity': 'CRITICAL'}, {'name': 'no_negative_rating', 'column': 'rating', 'condition': 'less_than', 'value': 0, 'severity': 'CRITICAL'}, {'name': 'rating_max_limit', 'column': 'rating', 'condition': 'greater_than', 'value': 5, 'severity': 'WARNING'}, {'name': 'missing_product_id', 'column': 'product_id', 'condition': 'is_null', 'value': None, 'severity': 'CRITICAL'}]


In [6]:
config = load_config("../config/pipeline_config.yaml")
print(config.get("validation_rules", "NO RULES FOUND"))

[{'name': 'no_negative_price', 'column': 'initial_price', 'condition': 'less_than', 'value': 0, 'severity': 'CRITICAL'}, {'name': 'no_negative_rating', 'column': 'rating', 'condition': 'less_than', 'value': 0, 'severity': 'CRITICAL'}, {'name': 'rating_max_limit', 'column': 'rating', 'condition': 'greater_than', 'value': 5, 'severity': 'WARNING'}, {'name': 'missing_product_id', 'column': 'product_id', 'condition': 'is_null', 'value': None, 'severity': 'CRITICAL'}]


In [7]:
test_df = df.copy()
test_df.loc[len(test_df)] = test_df.iloc[0]
test_df.loc[len(test_df)-1, "rating"] = -5
test_df.loc[len(test_df)-1, "initial_price"] = -100

violations_test, summary_test = apply_validation_rules(test_df, config)
clean_test, quarantine_test = quarantine_bad_rows(test_df, violations_test, config)

print(f"Quarantined: {len(quarantine_test)} rows")
print(quarantine_test[["product_id", "rating", "initial_price", "quarantine_reason"]])

Quarantined: 1 rows
    product_id  rating  initial_price  \
39    17352320    -5.0         -100.0   

                                    quarantine_reason  
39  no_negative_price (column: initial_price, valu...  


In [8]:
from src.anomaly_detector import detect_statistical_anomalies, detect_volume_anomaly
from src.schema_drift import save_schema_snapshot, detect_schema_drift

# 1. Statistical Anomaly Detection
anomalies = detect_statistical_anomalies(df, profile, config)
print(f"📊 Statistical Anomalies Found: {len(anomalies)}")
for a in anomalies[:5]:
    print(f"  Row {a['row_index']} | {a['column']} = {a['value']} | Z-score: {a['z_score']}")

# 2. Schema Drift Detection
df_dtypes = {col: str(df[col].dtype) for col in df.columns}
drift_result = detect_schema_drift(list(df.columns), df_dtypes, metadata["file_name"])
print(f"\n🔄 Schema Drift Check: {drift_result}")

# Snapshot save karo agle run ke liye
save_schema_snapshot(list(df.columns), df_dtypes, config, metadata["file_name"])
print("✅ Schema snapshot saved for future comparison")

📊 Statistical Anomalies Found: 7
  Row 6 | product_id = 919032.0 | Z-score: -3.31
  Row 4 | rating = 0.0 | Z-score: -3.24
  Row 24 | rating = 0.0 | Z-score: -3.24
  Row 31 | rating = 0.0 | Z-score: -3.24
  Row 11 | ratings_count = 994.0 | Z-score: 4.23

🔄 Schema Drift Check: {'drift_detected': False, 'is_first_run': False, 'added_columns': [], 'removed_columns': [], 'type_changes': [], 'checked_at': '2026-07-03 17:00:00.829902'}
✅ Schema snapshot saved for future comparison


In [9]:
# Simulate karo ki naya column add hua aur ek column gayab ho gaya
fake_new_columns = list(df.columns) + ["new_promo_column"]
fake_new_columns.remove("currency")

fake_dtypes = {col: str(df[col].dtype) for col in df.columns if col != "currency"}
fake_dtypes["new_promo_column"] = "object"

drift_test = detect_schema_drift(fake_new_columns, fake_dtypes, metadata["file_name"])
print(json.dumps(drift_test, indent=2))

{
  "drift_detected": true,
  "is_first_run": false,
  "added_columns": [
    "new_promo_column"
  ],
  "removed_columns": [
    "currency"
  ],
  "type_changes": [],
  "checked_at": "2026-07-03 17:00:01.296101"
}


In [10]:
from src.ai_explainer import generate_anomaly_explanation

# Apni API key yahan daalo
API_KEY = "your-api-key-here"

if anomalies:
    sample_anomaly = anomalies[0]
    explanation = generate_anomaly_explanation(sample_anomaly, API_KEY)
    print("🤖 AI Explanation:")
    print(explanation)
else:
    print("Koi anomaly nahi mili test karne ke liye — pehle test_df wala anomaly inject karo")

🤖 AI Explanation:
AI explanation unavailable: 'content'


In [11]:
from src.azure_sql import save_pipeline_run
from src.quality_trend import get_quality_trend

# Config reload karo
config = load_config("../config/pipeline_config.yaml")

# Pipeline run SQLite mein save karo
run_record = save_pipeline_run(
    metadata=metadata,
    profile=profile,
    validation_summary=summary,
    anomalies=anomalies,
    drift_result=drift_result,
    config=config
)

print("✅ Run saved:")
for k, v in run_record.items():
    print(f"  {k}: {v}")

# Quality trend fetch karo
trend_df = get_quality_trend(config)
print(f"\n📈 Total runs in database: {len(trend_df)}")
print(trend_df)

✅ Run saved:
  file_name: tshirts.csv
  file_format: csv
  row_count: 39
  column_count: 24
  quality_score: 87.5
  critical_violations: 0
  warning_violations: 0
  anomalies_found: 7
  schema_drift_detected: 0
  run_timestamp: 2026-07-03 17:01:32.242141

📈 Total runs in database: 1
                run_timestamp    file_name  quality_score  anomalies_found  \
0  2026-07-03 17:01:32.242141  tshirts.csv           87.5                7   

   critical_violations  schema_drift_detected  
0                    0                      0  


In [2]:
import os
db_path = os.path.abspath("../logs/datasentinel.db")
print("DB path:", db_path)
print("DB exists:", os.path.exists(db_path))

DB path: C:\Users\Manoj Pareek\OneDrive\Desktop\DataSentinel\logs\datasentinel.db
DB exists: True
